# Exploratory: why heavy tails and why volatility scaling?

Scratch exploration behind the modelling choices in `src/` and `experiments/`.
Not part of the reproducible pipeline — anything load-bearing lives in a tested
module and is driven from an experiment script. Uses the cached data snapshot,
so run an experiment once (or `--refresh`) first to populate `results/`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, str(Path.cwd().parent))
from src import data as D
from src.plotstyle import use_house_style
use_house_style()

## Load BTC daily log returns

In [ ]:
prices = D.fetch_prices('BTC-USD', start='2015-01-01', end='2026-07-01')
returns = D.compute_log_returns(prices['Close'])
print(f'{len(returns)} daily observations, {returns.index.min().date()} to {returns.index.max().date()}')
print(f'excess kurtosis = {stats.kurtosis(returns):.2f}   (Gaussian = 0)')

## Volatility clustering

Big moves cluster in time. A 30-day rolling standard deviation makes the regime
structure obvious — this is what motivates the EWMA-scaled Gaussian model.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax1.plot(returns.index, returns.values, lw=0.6, color='#4C72B0')
ax1.set_ylabel('log return')
ax2.plot(returns.index, returns.rolling(30).std(), color='#C44E52')
ax2.set_ylabel('30d rolling std')
fig.tight_layout()

## Autocorrelation: returns vs. |returns|

Returns themselves are nearly uncorrelated (why the AR mean model is weak), but
|returns| are strongly autocorrelated — the signature of volatility clustering,
and the reason the |r| feature is included.

In [ ]:
def acf(x, lags):
    x = np.asarray(x) - np.mean(x)
    denom = np.dot(x, x)
    return [1.0] + [np.dot(x[:-k], x[k:]) / denom for k in range(1, lags + 1)]

lags = 25
fig, ax = plt.subplots()
ax.bar(np.arange(lags + 1) - 0.2, acf(returns, lags), 0.4, label='returns', color='#4C72B0')
ax.bar(np.arange(lags + 1) + 0.2, acf(returns.abs(), lags), 0.4, label='|returns|', color='#DD8452')
ax.axhline(0, color='#888', lw=0.8)
ax.set_xlabel('lag'); ax.set_ylabel('autocorrelation'); ax.legend()